### 1.Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application. 

## Driver
- Creates the SparkSession and starts the Spark application.
- Converts user code into Spark jobs and stages.
- Builds and optimizes the Directed Acyclic Graph (DAG).
- Schedules tasks and collects results from executors.

## Cluster Manager
- Allocates CPU, memory, and other cluster resources.
- Launches and manages executor processes on worker nodes.
- Monitors resource usage and application execution.
- Communicates with the driver to assign resources.
- Examples include Standalone, YARN, Kubernetes, and Mesos.

## Executor
- Executes tasks assigned by the driver.
- Processes data partitions in parallel.
- Stores cached or persisted data for faster processing.
- Sends computation results and task status back to the driver.

### 2.How does Spark's Lazy Evaluation improve performance?

Apache Spark uses a **Lazy Evaluation** strategy to optimize the execution of data processing tasks. Instead of executing each transformation immediately, Spark records all transformations, such as `filter()`, `select()`, `withColumn()`, and `groupBy()`, in a **Directed Acyclic Graph (DAG)**. This graph represents the sequence of operations required to produce the final result without actually processing the data at each step.

Spark postpones execution until an **action** such as `show()`, `count()`, `collect()`, or `write()` is invoked. At that point, Spark analyzes the entire DAG and creates an optimized physical execution plan. During this optimization process, Spark combines multiple transformations into a single stage whenever possible, eliminates unnecessary intermediate computations, and reduces redundant data scans.

This approach significantly enhances the performance and scalability of Spark applications, making it highly suitable for processing large-scale datasets in distributed computing environments.

### 3. Read a CSV file with header and inferSchema enabled.

In [0]:
df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/w6/default/week-6")
df.show()

+--------+-------+----------------+------------------+----------+--------------+-----------+----------+----------+---------+-------+-----------+--------+----------+
|order_id|user_id|   customer_name|          old_name|product_id|      category|      price|base_price|    status|   amount| region|       city|priority|order_date|
+--------+-------+----------------+------------------+----------+--------------+-----------+----------+----------+---------+-------+-----------+--------+----------+
|ORD10001|USR1409|     Navya Menon|     Women's Kurti|    PCL792|      Clothing| Rs.4468.94|   4468.94|   Pending| 10700.12|  North|      Delhi|     Low|2025-09-20|
|ORD10002|   NULL|    Saanvi Yadav|        Table Lamp|    PHO703|Home & Kitchen| Rs.1740.33|   1740.33|Processing|  4957.82|  North|     Jaipur|     Low|2025-03-07|
|ORD10003|USR6514|   Reyansh Reddy|       Curtain Set|    PHO967|Home & Kitchen| Rs.2128.34|   2128.34| Completed|  1952.36|  South|  Hyderabad|Critical|2025-07-03|
|ORD10004|

### 4.What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

## - CSV:
- Row-based storage
- Larger file size
- Slower read performance
- Does not store schema
## - Parquet:
- Column-based storage
- Compressed file size
- Faster read performance
- Stores schema
- Reads only the required columns
### why does it matter for Performance?
- Columnar storage allows Parquet to read only the required columns.
- Compression reduces storage space and speeds up data transfer.
- Less disk I/O because only relevant data is accessed.
- Predicate pushdown filters data while reading, avoiding unnecessary scans.

### 5.Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'. 

In [0]:
df.filter(df.category=="Electronics").select("product_id","price").show()

+----------+-----------+
|product_id|      price|
+----------+-----------+
|    PEL651|Rs.24014.37|
|    PEL261|Rs.41107.27|
|    PEL652|Rs.85837.23|
|    PEL335|Rs.53164.39|
|    PEL919|Rs.77618.69|
|    PEL956| Rs.1819.83|
|    PEL371|Rs.83422.37|
|    PEL588|Rs.45504.11|
|    PEL568|Rs.28796.86|
|    PEL860|Rs.50012.52|
|    PEL766|Rs.24671.37|
|    PEL514|Rs.62413.95|
|    PEL581|Rs.46946.26|
|    PEL918|Rs.21916.12|
|    PEL129| Rs.4615.19|
|    PEL734|Rs.61639.54|
|    PEL453|Rs.88893.83|
|    PEL908|Rs.71063.62|
|    PEL973|Rs.59095.81|
|    PEL675|Rs.30002.32|
+----------+-----------+
only showing top 20 rows


### 6.Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double. 

In [0]:
df.select("price").show(10,False)
##To know how the data is stored in price coumn
df.select("price").schema

+-----------+
|price      |
+-----------+
|Rs.4468.94 |
|Rs.1740.33 |
|Rs.2128.34 |
|Rs.1443.6  |
|Rs.5625.26 |
|Rs.1960.17 |
|Rs.24014.37|
|Rs.3617.81 |
|Rs.41107.27|
|Rs.85837.23|
+-----------+
only showing top 10 rows


StructType([StructField('price', StringType(), True)])

In [0]:
from pyspark.sql.functions import *
df2=df.withColumnRenamed("old_name","new_name")
df2=df.withColumn("price",regexp_replace(col("price"),"Rs\\.","").cast("double"))
df2.show()

+--------+-------+----------------+------------------+----------+--------------+--------+----------+----------+---------+-------+-----------+--------+----------+
|order_id|user_id|   customer_name|          old_name|product_id|      category|   price|base_price|    status|   amount| region|       city|priority|order_date|
+--------+-------+----------------+------------------+----------+--------------+--------+----------+----------+---------+-------+-----------+--------+----------+
|ORD10001|USR1409|     Navya Menon|     Women's Kurti|    PCL792|      Clothing| 4468.94|   4468.94|   Pending| 10700.12|  North|      Delhi|     Low|2025-09-20|
|ORD10002|   NULL|    Saanvi Yadav|        Table Lamp|    PHO703|Home & Kitchen| 1740.33|   1740.33|Processing|  4957.82|  North|     Jaipur|     Low|2025-03-07|
|ORD10003|USR6514|   Reyansh Reddy|       Curtain Set|    PHO967|Home & Kitchen| 2128.34|   2128.34| Completed|  1952.36|  South|  Hyderabad|Critical|2025-07-03|
|ORD10004|USR6925|     Navee

### 7.How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Apache Spark provides fault tolerance through its **Lineage Graph**. Instead of storing multiple copies of intermediate data, Spark records every transformation performed on the original dataset, such as `filter()`, `select()`, `map()`, and `groupBy()`. This sequence of transformations forms the lineage of the data.

When a worker node or executor fails, the partitions stored on that node may be lost. Rather than restarting the entire application, Spark uses the lineage information stored in the DAG to identify only the lost partitions and recomputes them by reapplying the required transformations on the original data. The remaining successfully computed partitions are reused, avoiding unnecessary recomputation.

This lineage-based recovery mechanism makes Spark highly fault tolerant while reducing storage overhead compared to systems that rely on data replication. As a result, Spark can efficiently recover from node failures, maintain reliable execution, and continue processing large-scale distributed datasets with minimal performance impact.

### 8.Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 

In [0]:
df3=df.filter((df.status=="Completed")&(df.amount>1000))
df3.show()

+--------+-------+----------------+------------------+----------+--------------+-----------+----------+---------+---------+-------+----------+--------+----------+
|order_id|user_id|   customer_name|          old_name|product_id|      category|      price|base_price|   status|   amount| region|      city|priority|order_date|
+--------+-------+----------------+------------------+----------+--------------+-----------+----------+---------+---------+-------+----------+--------+----------+
|ORD10003|USR6514|   Reyansh Reddy|       Curtain Set|    PHO967|Home & Kitchen| Rs.2128.34|   2128.34|Completed|  1952.36|  South| Hyderabad|Critical|2025-07-03|
|ORD10004|USR6925|     Naveen Nair|Remote Control Car|    PTO777|          Toys|  Rs.1443.6|    1443.6|Completed|  1465.89|  North|     Delhi|  Medium|2025-08-21|
|ORD10006|USR4598|     Rahul Verma|     Men's T-Shirt|    PCL924|      Clothing| Rs.1960.17|   1960.17|Completed|  2684.75|Central|    Raipur|     Low|2025-08-13|
|ORD10008|USR1771|   K

### 9.Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Predicate Pushdown is a performance optimization technique used by Spark when working with columnar file formats such as Parquet and ORC. When a filter condition is applied, Spark sends the condition to the storage layer before reading the data, instead of loading the entire dataset into memory.

Parquet stores metadata for each row group, including information such as the minimum and maximum values of columns. Spark uses this metadata to identify which row groups satisfy the filter condition and skips the remaining row groups that are not required. As a result, only the relevant data is read and processed.

This approach significantly reduces the amount of data loaded into memory, minimizes disk I/O operations, and decreases network communication between storage and executors. Since Spark processes only the necessary data, query execution becomes faster and overall resource utilization improves.

Predicate Pushdown is particularly beneficial when working with large datasets, as it enhances the performance and scalability of Spark applications while reducing execution time and memory consumption.

### 10.Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [0]:
df4=df.withColumn("final_price",col("base_price")*1.18)
df.show()

+--------+-------+----------------+------------------+----------+--------------+-----------+----------+----------+---------+-------+-----------+--------+----------+
|order_id|user_id|   customer_name|          old_name|product_id|      category|      price|base_price|    status|   amount| region|       city|priority|order_date|
+--------+-------+----------------+------------------+----------+--------------+-----------+----------+----------+---------+-------+-----------+--------+----------+
|ORD10001|USR1409|     Navya Menon|     Women's Kurti|    PCL792|      Clothing| Rs.4468.94|   4468.94|   Pending| 10700.12|  North|      Delhi|     Low|2025-09-20|
|ORD10002|   NULL|    Saanvi Yadav|        Table Lamp|    PHO703|Home & Kitchen| Rs.1740.33|   1740.33|Processing|  4957.82|  North|     Jaipur|     Low|2025-03-07|
|ORD10003|USR6514|   Reyansh Reddy|       Curtain Set|    PHO967|Home & Kitchen| Rs.2128.34|   2128.34| Completed|  1952.36|  South|  Hyderabad|Critical|2025-07-03|
|ORD10004|

### 11.What is the difference between Transformations and Actions? Provide two examples of each.

### Transformations
- Create a new DataFrame or modify an existing DataFrame.
- Lazy in nature (not executed immediately).
- Build the execution plan (DAG).
- Return another DataFrame.
### - Examples:
- df.filter(df.price > 500)
- df.select("name", "price")
- df.withColumn("discount", col("price") * 0.1)
### Actions
- Trigger the execution of transformations.
- Executed immediately.
- Return results or perform an output operation.
- Return a value or display the output.
### - Examples:
- df.show()
- df.count()
- df.collect()

### 12.Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output". 

In [0]:
input_df=spark.read.parquet("path/to/input")
clean_df=input_df.filter(input_df.user_id.isNotNull())
clean_df.write.mode("overwrite").option("header", True).csv("path/to/output")

In [0]:
clean_df=df.filter(df.user_id.isNotNull())
clean_df.show(10)

+--------+-------+-------------+------------------+----------+--------------+-----------+----------+----------+---------+-------+-----------+--------+----------+
|order_id|user_id|customer_name|          old_name|product_id|      category|      price|base_price|    status|   amount| region|       city|priority|order_date|
+--------+-------+-------------+------------------+----------+--------------+-----------+----------+----------+---------+-------+-----------+--------+----------+
|ORD10001|USR1409|  Navya Menon|     Women's Kurti|    PCL792|      Clothing| Rs.4468.94|   4468.94|   Pending| 10700.12|  North|      Delhi|     Low|2025-09-20|
|ORD10003|USR6514|Reyansh Reddy|       Curtain Set|    PHO967|Home & Kitchen| Rs.2128.34|   2128.34| Completed|  1952.36|  South|  Hyderabad|Critical|2025-07-03|
|ORD10004|USR6925|  Naveen Nair|Remote Control Car|    PTO777|          Toys|  Rs.1443.6|    1443.6| Completed|  1465.89|  North|      Delhi|  Medium|2025-08-21|
|ORD10005|USR3664|  Sneha Si

### 13.In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

#### Client Mode
- The Driver program runs on the client machine from which the Spark application is submitted.
- Executors run on the worker nodes in the cluster and communicate with the Driver.
- Suitable for development, testing, and interactive analysis where the user needs direct access to the application.
- If the client machine disconnects or fails, the Spark application also terminates because the Driver is no longer available.

#### Cluster Mode
- The Driver program runs inside the cluster on one of the worker nodes.
- Both the Driver and Executors are managed by the Cluster Manager.
- Suitable for production environments and long-running Spark applications.
- The application continues to execute even if the client disconnects after submitting the job, making it more reliable and fault tolerant.

### 14.Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [0]:
df.filter((df.region=="North")|(df.priority=="High")).show()

+--------+-------+---------------+------------------+----------+--------------+-----------+----------+----------+---------+-------+----------+--------+----------+
|order_id|user_id|  customer_name|          old_name|product_id|      category|      price|base_price|    status|   amount| region|      city|priority|order_date|
+--------+-------+---------------+------------------+----------+--------------+-----------+----------+----------+---------+-------+----------+--------+----------+
|ORD10001|USR1409|    Navya Menon|     Women's Kurti|    PCL792|      Clothing| Rs.4468.94|   4468.94|   Pending| 10700.12|  North|     Delhi|     Low|2025-09-20|
|ORD10002|   NULL|   Saanvi Yadav|        Table Lamp|    PHO703|Home & Kitchen| Rs.1740.33|   1740.33|Processing|  4957.82|  North|    Jaipur|     Low|2025-03-07|
|ORD10004|USR6925|    Naveen Nair|Remote Control Car|    PTO777|          Toys|  Rs.1443.6|    1443.6| Completed|  1465.89|  North|     Delhi|  Medium|2025-08-21|
|ORD10005|USR3664|    

### 15.When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

The `show(5)` method displays only the first five rows of a DataFrame, making it an efficient and safe way to inspect data during development and analysis. It retrieves only a small portion of the dataset, allowing users to quickly verify the structure, schema, and sample records without consuming significant system resources.

In contrast, the `collect()` method retrieves the entire dataset from all executors and transfers it to the Driver's memory. For very large datasets, such as those containing millions of records or several terabytes of data, this can result in excessive memory consumption, increased network traffic, and may even cause an **OutOfMemoryError** on the Driver node.

Therefore, `show(5)` is the preferred choice for exploring large datasets because it is memory-efficient, faster, and avoids unnecessary data transfer, making Spark applications more scalable and reliable.